# 🦀 Day 3: Introduction to Axum Web Framework

---

## What is Axum?

**Axum** is an ergonomic web framework built on Tokio. It provides:

- Type-safe routing and extractors
- Async handlers out of the box
- Integration with Tower middleware
- First-class support for JSON (serde)

Axum is designed for building APIs and web services. It's the framework of choice for many Rust web projects.

In [ ]:
:dep axum = "0.7"
:dep tokio = { version = "1", features = ["full"] }
:dep serde = { version = "1", features = ["derive"] }
:dep serde_json = "1"

// Dependencies loaded. Axum requires a long-running server — we'll show the structure here.

## Basic Server Setup with Router

Axum uses a `Router` to define routes. You build the router, then serve it with a listener.

In [ ]:
use axum::{Router, routing::get};

// A route handler — async function that returns a response
async fn hello_handler() -> &'static str {
    "Hello, Axum!"
}

// Build the router (this works in EvCxR — we're not starting the server)
let app: Router = Router::new().route("/", get(hello_handler));

println!("Router built. In a real binary, we would run: tokio::net::TcpListener::bind(...).await");

## Route Handlers as Async Functions

Handlers are `async fn` that return types implementing `IntoResponse`. Common return types: `String`, `&'static str`, `Json<T>`, `StatusCode`.

In [ ]:
use axum::{Json, http::StatusCode};
use serde::Serialize;

#[derive(Serialize)]
struct Greeting {
    message: String,
    version: u32,
}

async fn json_handler() -> Json<Greeting> {
    Json(Greeting {
        message: "Hello from Axum!".to_string(),
        version: 1,
    })
}

async fn status_handler() -> StatusCode {
    StatusCode::OK
}

// We can build a router with these handlers
let _app = Router::new()
    .route("/json", get(json_handler))
    .route("/health", get(status_handler));

println!("Handlers defined. Json<T> and StatusCode both implement IntoResponse.");

## GET and POST Handlers

Use `get()` and `post()` from `axum::routing` to register handlers.

In [ ]:
use axum::{Router, routing::{get, post}, Json};
use serde::{Deserialize, Serialize};

#[derive(Deserialize)]
struct CreateUser {
    name: String,
    email: String,
}

#[derive(Serialize)]
struct UserResponse {
    id: u32,
    name: String,
    email: String,
}

async fn get_users() -> Json<Vec<UserResponse>> {
    Json(vec![])
}

async fn create_user(Json(payload): Json<CreateUser>) -> Json<UserResponse> {
    Json(UserResponse {
        id: 1,
        name: payload.name,
        email: payload.email,
    })
}

let app = Router::new()
    .route("/users", get(get_users).post(create_user));

println!("Router with GET and POST for /users");

## Running the Server (Cargo Binary)

A server runs indefinitely, so it **cannot run inside an EvCxR notebook cell**. Here's the complete code you would use in a `main.rs` for a Cargo project:

In [ ]:
// === Copy this into src/main.rs of a Cargo project ===
//
// [dependencies]
// axum = "0.7"
// tokio = { version = "1", features = ["full"] }
// serde = { version = "1", features = ["derive"] }
// serde_json = "1"
//
// use axum::{Router, routing::get, Json};
// use serde::Serialize;
//
// #[derive(Serialize)]
// struct Hello { msg: String }
//
// async fn root() -> &'static str { "Hello, World!" }
// async fn api() -> Json<Hello> { Json(Hello { msg: "Hi from API".into() }) }
//
// #[tokio::main]
// async fn main() {
//     let app = Router::new()
//         .route("/", get(root))
//         .route("/api", get(api));
//     let listener = tokio::net::TcpListener::bind("127.0.0.1:3000").await.unwrap();
//     axum::serve(listener, app).await.unwrap();
// }
//
// Then: cargo run, and visit http://127.0.0.1:3000

println!("See comments above for full server code.");

## Returning Different Response Types

| Type | Use Case |
|------|----------|
| `&'static str` / `String` | Plain text |
| `Json<T>` | JSON response (sets Content-Type) |
| `StatusCode` | Empty body, e.g. 204 No Content |
| `(StatusCode, T)` | Status + body |
| `Response` | Full control |

All of these implement `IntoResponse`.

In [ ]:
use axum::{Json, http::StatusCode, response::IntoResponse};

// (StatusCode, String) for custom status + body
async fn error_handler() -> (StatusCode, String) {
    (StatusCode::NOT_FOUND, "Resource not found".to_string())
}

// (StatusCode, Json<T>) for JSON error responses
async fn json_error() -> (StatusCode, Json<serde_json::Value>) {
    (
        StatusCode::BAD_REQUEST,
        Json(serde_json::json!({ "error": "Invalid input" })),
    )
}

let _ = (error_handler, json_error);
println!("Handlers returning (StatusCode, body) for error responses.");

## 🎯 Key Takeaways

1. **Axum** is an ergonomic web framework built on Tokio
2. `Router::new().route("/path", get(handler))` for routing
3. Handlers are `async fn` returning `IntoResponse` (String, Json, StatusCode, etc.)
4. `get()` and `post()` from `axum::routing`
5. `Json<T>` for JSON responses; `(StatusCode, T)` for status + body
6. Server runs with `axum::serve(listener, app)` — use a Cargo binary, not a notebook

---

**Tomorrow:** Request handling — path params, query params, JSON bodies! 📥